# 🛠️ Ollama & Hugging Face Model-Download Utility
**Projekt: Mai_AI (MaiOmni) — Dynamisches Dienstprogramm zur Modellverwaltung**

Dieses Notebook dient als **schlankes, interaktives Dienstprogramm** für das dynamische Abfragen, Suchen/Filtern, Herunterladen und automatische Einbinden von Offline-Modellen in deine lokale **Ollama-Laufzeitumgebung**.

---
### 🎯 Ablauf:
1. **Plattform-Auswahl:** Wähle die Plattform (**Ollama** oder **Hugging Face**) und bestätige mit **„OK - Plattform bestätigen“**.
2. **Live-Filter & Modellauswahl:** Nutze das **Such- und Filterfeld**, um die Modelle in Echtzeit nach Namen, Aufgabenbereichen oder Beschreibungen zu durchsuchen, und wähle dein Modell im Dropdown-Menü.
3. **Download & Einbindung:** Nach Klick auf **„OK - Download & In Ollama einbinden“** erfolgt der Download und die automatische Registrierung im lokalen Ollama-Dienst vollkommen autonom im Hintergrund.

---

### ⚙️ Schritt 1: System- & Dienst-Initialisierung
Überprüfung aller erforderlichen Python-Abhängigkeiten und plattformübergreifender Health-Check des lokalen Ollama-Dienstes.

In [ ]:
# Absicherung der Konsolenausgabe gegen Windows-Encoding-Fehler (cp1252)
import sys
if sys.stdout and hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

import os
import json
import subprocess
import importlib
import platform
import time
import shutil
import urllib.request
import psutil

# 1. Selbstheilende Abhängigkeitsprüfung
required_packages = {
    "ipywidgets": "ipywidgets",
    "ollama": "ollama",
    "psutil": "psutil",
    "huggingface_hub": "huggingface_hub",
    "tqdm": "tqdm"
}

missing_packages = []
for module_name, package_name in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing_packages.append(package_name)

if missing_packages:
    print(f"[AUTO-REPARATUR] Installiere fehlende Pakete: {missing_packages}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print("[✓] Abhängigkeiten erfolgreich bereitgestellt.")

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import ollama
from ollama import Client
from huggingface_hub import HfApi, hf_hub_download

# 2. Projektpfade ermitteln
def get_project_root() -> str:
    if "__file__" in globals():
        current_path = os.path.abspath(os.path.dirname(__file__))
    else:
        current_path = os.path.abspath(os.getcwd())
    
    path_parts = current_path.split(os.sep)
    if "Offline_AI" in path_parts:
        root_index = path_parts.index("Offline_AI")
        if os.name == 'nt':
            root_path = path_parts[0] + os.sep + os.path.join(*path_parts[1:root_index + 1])
        else:
            root_path = os.sep + os.path.join(*path_parts[:root_index + 1])
        return os.path.abspath(root_path)
    return current_path

PROJECT_ROOT = get_project_root()
CONFIG_DIR = os.path.join(PROJECT_ROOT, "config")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
HF_MODELS_DIR = os.path.join(DATA_DIR, "models", "huggingface")
os.makedirs(CONFIG_DIR, exist_ok=True)
os.makedirs(HF_MODELS_DIR, exist_ok=True)

# 3. Hardware-Spezifikationen ermitteln
TOTAL_RAM_GB = round(psutil.virtual_memory().total / (1024**3), 2)
SAFE_RAM_BUDGET_GB = round(TOTAL_RAM_GB * 0.70, 1)
CPU_COUNT = psutil.cpu_count(logical=True)

# 4. Plattformunabhängiger Ollama-Service-Manager
class OllamaServiceManager:
    def __init__(self, host="127.0.0.1", port=11434):
        self.host = host
        self.port = port
        self.client = Client(host=f"http://{host}:{port}")

    def is_running(self, timeout=1.5) -> bool:
        try:
            Client(host=f"http://{self.host}:{self.port}", timeout=timeout).list()
            return True
        except Exception:
            return False

    def ensure_service(self, max_wait_seconds=20) -> bool:
        if self.is_running():
            return True
        
        print("[WARNUNG] Ollama-Dienst nicht aktiv. Starte Hintergrundprozess...")
        current_os = platform.system()
        if current_os == "Windows":
            candidates = ["ollama.exe", os.path.expanduser("~\\AppData\\Local\\Programs\\Ollama\\ollama.exe")]
            for cand in candidates:
                found = shutil.which(cand) or (cand if os.path.exists(cand) else None)
                if found:
                    creationflags = subprocess.CREATE_NEW_PROCESS_GROUP if hasattr(subprocess, 'CREATE_NEW_PROCESS_GROUP') else 0
                    subprocess.Popen([found, "serve"], creationflags=creationflags, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                    break
        elif current_os == "Darwin":
            try:
                subprocess.Popen(["open", "-a", "Ollama"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            except Exception:
                subprocess.Popen(["ollama", "serve"], start_new_session=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        else:
            try:
                subprocess.run(["systemctl", "start", "ollama"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            except Exception:
                subprocess.Popen(["ollama", "serve"], start_new_session=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        start_time = time.time()
        while time.time() - start_time < max_wait_seconds:
            time.sleep(1)
            if self.is_running():
                print("[✓] Ollama-Dienst erfolgreich verbunden!")
                return True
        return False

service_manager = OllamaServiceManager()
ollama_ready = service_manager.ensure_service()

print("=" * 65)
print(f"[✓] SYSTEM-STATUS: {platform.system()} ({platform.machine()})")
print(f"    -> CPU: {CPU_COUNT} Kerne | RAM: {TOTAL_RAM_GB} GB (Budget: {SAFE_RAM_BUDGET_GB} GB)")
print(f"    -> Ollama-Dienst: {'[✓] Verbunden' if ollama_ready else '[!] Nicht erreichbar'}")
print("=" * 65)

### 🔌 Schritt 2: Backend-Logik & Dynamische API-Schnittstellen
Trennung der Datenbeschaffung von der Benutzeroberfläche: Die Klassen `OllamaBackend` und `HuggingFaceBackend` rufen Modelle live ohne Hardcoding ab und steuern den Download-Prozess.

In [ ]:
# ==============================================================================
# BACKEND: OLLAMA API & LIVE LIBRARY FETCHER
# ==============================================================================
class OllamaBackend:
    ONLINE_LIBRARY_URL = "https://raw.githubusercontent.com/chrizzo84/OllamaScraper/refs/heads/main/out/ollama_models.json"

    def __init__(self, host="127.0.0.1", port=11434):
        self.host = host
        self.port = port
        self.client = Client(host=f"http://{host}:{port}")

    def fetch_models(self, limit=80) -> list[dict]:
        """Ruft die verfügbaren Modelle dynamisch aus der offiziellen Library ab (Fallback: lokaler Daemon)."""
        models = []
        try:
            req = urllib.request.Request(self.ONLINE_LIBRARY_URL, headers={"User-Agent": "Offline_AI/1.0"})
            with urllib.request.urlopen(req, timeout=4.0) as response:
                data = json.loads(response.read().decode("utf-8"))
                raw_models = data.get("models", []) if isinstance(data, dict) else (data if isinstance(data, list) else [])
                
                for item in raw_models[:limit]:
                    name = item.get("name", "")
                    pulls = item.get("pulls_text", "")
                    capabilities = item.get("capabilities", [])
                    variants = item.get("variants", [])
                    blurb = item.get("blurb", "") or item.get("description", "")
                    category = ", ".join(capabilities) if capabilities else "Allgemein / LLM"
                    
                    if variants:
                        for var in variants[:4]:
                            tag = var.get("tag", name)
                            size_text = var.get("size_text", "Variabel")
                            models.append({
                                "id": tag,
                                "name": tag,
                                "size": size_text,
                                "category": category,
                                "description": blurb,
                                "popularity": pulls,
                                "platform": "ollama"
                            })
                    else:
                        models.append({
                            "id": name,
                            "name": name,
                            "size": "Variabel",
                            "category": category,
                            "description": blurb,
                            "popularity": pulls,
                            "platform": "ollama"
                        })
                if models:
                    return models
        except Exception as err:
            print(f"[Hinweis] Online-Bibliothek nicht erreichbar ({err}). Nutze lokalen Ollama-Daemon...")

        # Fallback: Lokale Tags über API
        try:
            local_tags = self.client.list()
            for m in local_tags.models:
                size_mb = m.size // (1024**2) if m.size else 0
                family = getattr(m.details, "family", "lokal") or "lokal"
                models.append({
                    "id": m.model,
                    "name": m.model,
                    "size": f"{size_mb} MB",
                    "category": family,
                    "description": f"Lokal installiertes Modell ({family})",
                    "popularity": "Lokal vorhanden",
                    "platform": "ollama"
                })
        except Exception as err:
            print(f"[Fehler] Lokale Modelle konnten nicht abgefragt werden: {err}")

        return models

    def pull_model(self, model_tag: str, progress_callback=None):
        """Führt den Streaming-Pull eines Modells über den Ollama Client aus."""
        for progress in self.client.pull(model=model_tag, stream=True):
            if progress_callback:
                progress_callback(progress)

# ==============================================================================
# BACKEND: HUGGING FACE API & GGUF INTEGRATION
# ==============================================================================
class HuggingFaceBackend:
    def __init__(self):
        self.api = HfApi()

    def fetch_models(self, limit=50) -> list[dict]:
        """Ruft die beliebtesten GGUF-Modelle dynamisch von Hugging Face ab."""
        models = []
        try:
            results = self.api.list_models(
                filter="gguf",
                sort="downloads",
                limit=limit
            )
            for m in results:
                downloads = getattr(m, "downloads", 0)
                pipeline = getattr(m, "pipeline_tag", "text-generation") or "text-generation"
                models.append({
                    "id": m.id,
                    "name": m.id,
                    "size": "GGUF Repository",
                    "category": pipeline,
                    "description": f"Hugging Face GGUF Repository: {m.id} | Task: {pipeline}",
                    "popularity": f"{downloads:,} Pull Downloads",
                    "platform": "huggingface"
                })
        except Exception as err:
            print(f"[Fehler] Hugging Face API-Abruf fehlgeschlagen: {err}")
        return models

    def fetch_gguf_files(self, repo_id: str) -> list[str]:
        """Ruft alle .gguf-Dateien eines Repositories dynamisch ab."""
        try:
            files = self.api.list_repo_files(repo_id=repo_id)
            gguf_files = [f for f in files if f.lower().endswith(".gguf")]
            return sorted(gguf_files, key=lambda x: (not ("q4_k_m" in x.lower() or "q4_0" in x.lower()), x))
        except Exception as err:
            print(f"[Fehler] Dateiliste für '{repo_id}' konnte nicht geladen werden: {err}")
            return []

    def download_and_create_model(self, repo_id: str, filename: str, target_model_tag: str, dest_dir: str, progress_callback=None, build_callback=None):
        """Lädt eine GGUF-Datei herunter und bindet sie automatisch in Ollama ein."""
        if progress_callback:
            progress_callback(f"Lade '{filename}' von Hugging Face ({repo_id}) herunter...")
        
        local_path = hf_hub_download(repo_id=repo_id, filename=filename, local_dir=dest_dir)
        abs_local_path = os.path.abspath(local_path)
        
        if progress_callback:
            progress_callback(f"GGUF gesichert. Registriere Modell '{target_model_tag}' in Ollama...")
        
        ollama_client = Client()
        for resp in ollama_client.create(model=target_model_tag, from_=abs_local_path, stream=True):
            if build_callback:
                build_callback(resp.get('status', ''))
        
        file_size_gb = round(os.path.getsize(abs_local_path) / (1024**3), 2)
        return abs_local_path, file_size_gb

# ==============================================================================
# CONFIG MANAGER
# ==============================================================================
class ModelConfigManager:
    @staticmethod
    def save_active_model(model_name: str, size_gb: float, platform_name: str, config_dir: str):
        config_path = os.path.join(config_dir, "active_model_config.json")
        data = {
            "model_name": model_name,
            "allocated_size_gb": size_gb,
            "detected_ram_gb": TOTAL_RAM_GB,
            "platform": platform_name,
            "updated_at": time.strftime('%Y-%m-%d %H:%M:%S')
        }
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4, ensure_ascii=False)

ollama_backend = OllamaBackend()
hf_backend = HuggingFaceBackend()
print("[✓] Backend-Module erfolgreich initialisiert.")

### 🎛️ Schritt 3: Interaktive Benutzeroberfläche (`ipywidgets`)
Führe die Zelle aus, um das interaktive Cockpit zu starten:
1. Plattform wählen (**Ollama** oder **Hugging Face**) und mit **„OK - Plattform bestätigen“** die dynamische Liste abrufen.
2. Nutze das **Suchfeld**, um Modelle in Echtzeit nach Namen, Aufgabenbereichen oder Beschreibungen zu filtern, und wähle dein Modell.
3. Mit **„OK - Download & In Ollama einbinden“** den Download und die Integration ausführen.

In [ ]:
import threading

# ==============================================================================
# UI WIDGETS
# ==============================================================================

# Dropdown 1: Plattform-Auswahl
dropdown_platform = widgets.Dropdown(
    options=[
        ("🦙 Ollama (Offizielle Library & Live-Registry)", "ollama"),
        ("🤗 Hugging Face (GGUF Community Modelle - Top Downloads)", "huggingface")
    ],
    value="ollama",
    description="1. Plattform:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="480px")
)

# Button 1: OK - Plattform bestätigen
btn_confirm_platform = widgets.Button(
    description="OK - Plattform bestätigen",
    button_style="info",
    icon="check",
    layout=widgets.Layout(width="200px", margin="0 0 0 10px")
)

# Such- und Filter-Textfeld (Echtzeit-Filterung vor Dropdown 2)
text_model_search = widgets.Text(
    placeholder="Suchbegriff eingeben (z.B. coder, vision, 8b, qwen, llama, math...)",
    description="Modell suchen:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="680px")
)

# Dropdown 2: Dynamische Modell-Auswahl
dropdown_model = widgets.Dropdown(
    options=[],
    description="2. Modell:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="680px"),
    disabled=True
)

# Dropdown 3: GGUF-Dateiauswahl (für Hugging Face)
dropdown_gguf_file = widgets.Dropdown(
    options=[],
    description="GGUF-Datei:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="680px", display="none")
)

# Manuelle Eingabefelder (für benutzerdefinierte Modelle)
text_custom_tag = widgets.Text(
    placeholder="z.B. deepseek-coder:6.7b oder mistral-nemo",
    description="Ollama Tag:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="480px", display="none")
)

text_custom_hf_repo = widgets.Text(
    placeholder="z.B. TheBloke/Mistral-7B-Instruct-v0.2-GGUF",
    description="HF Repo ID:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="480px", display="none")
)

text_custom_hf_file = widgets.Text(
    placeholder="z.B. mistral-7b-instruct-v0.2.Q4_K_M.gguf",
    description="HF Dateiname:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="480px", display="none")
)

# Kompakte Info-Karte für technische Spezifikationen
html_spec_card = widgets.HTML(
    value="<div style='padding: 10px; border-left: 4px solid #3B82F6; background-color: #F9FAFB; border-radius: 4px; font-size: 13px;'>"\
          "Wähle eine Plattform und klicke auf <b>'OK - Plattform bestätigen'</b>, um Live-Modelle abzurufen.</div>"
)

# Button 2: OK - Download & Integration
btn_start_download = widgets.Button(
    description="OK - Download & In Ollama einbinden",
    button_style="success",
    icon="cloud-download",
    layout=widgets.Layout(width="300px", height="38px", margin="8px 0 0 0"),
    disabled=True
)

# Ladebalken & Statuszeile
progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Fortschritt:",
    bar_style="info",
    orientation="horizontal",
    style={"description_width": "110px", "bar_color": "#10B981"},
    layout=widgets.Layout(width="680px", display="none")
)

label_status = widgets.Label(
    value="Bereit.",
    layout=widgets.Layout(margin="4px 0 0 110px")
)

# Ausgabekonsole für detaillierte Logs
output_log = widgets.Output(
    layout=widgets.Layout(width="680px", height="160px", border="1px solid #E5E7EB", padding="8px", overflow="auto", background_color="#1E1E1E")
)

# ==============================================================================
# UI CONTROLLER & EVENT LOGIC
# ==============================================================================
CURRENT_LOADED_MODELS = []

def render_spec_card(model_item: dict) -> str:
    if not model_item:
        return "<div style='padding: 10px; color: #6B7280; font-size: 13px;'>Kein Modell ausgewählt.</div>"
    
    model_id = model_item.get("id", "-")
    size = model_item.get("size", "-")
    category = model_item.get("category", "-")
    popularity = model_item.get("popularity", "-")
    platform_name = model_item.get("platform", "").upper()
    
    html = f"""
    <div style='background: #FFFFFF; border: 1px solid #E5E7EB; border-radius: 6px; padding: 10px; margin-top: 6px;'>
        <div style='display: flex; justify-content: space-between; align-items: center; margin-bottom: 6px;'>
            <span style='background-color: #3B82F6; color: white; font-weight: bold; padding: 2px 8px; border-radius: 4px; font-size: 11px;'>{platform_name}</span>
            <span style='font-weight: bold; font-size: 13px; color: #111827;'>{model_id}</span>
            <span style='background-color: #10B981; color: white; padding: 2px 8px; border-radius: 4px; font-size: 11px;'>RAM verfügbar: {TOTAL_RAM_GB} GB</span>
        </div>
        <div style='display: grid; grid-template-columns: repeat(3, 1fr); gap: 6px; background: #F9FAFB; padding: 6px; border-radius: 4px; font-size: 12px; color: #374151;'>
            <div><b>Größe:</b> {size}</div>
            <div><b>Schwerpunkt/Task:</b> {category}</div>
            <div><b>Downloads/Pulls:</b> {popularity}</div>
        </div>
    </div>
    """
    return html

def filter_model_options(query: str = ""):
    """Filtert die geladenen Modelle in Echtzeit anhand des Suchbegriffs."""
    platform_type = dropdown_platform.value
    q = query.strip().lower()
    
    if not CURRENT_LOADED_MODELS:
        dropdown_model.options = [("Keine Modelle geladen", "")]
        dropdown_model.disabled = True
        btn_start_download.disabled = True
        return
    
    matched_models = []
    for m in CURRENT_LOADED_MODELS:
        searchable_content = f"{m.get('name', '')} {m.get('id', '')} {m.get('category', '')} {m.get('size', '')} {m.get('description', '')} {m.get('popularity', '')}".lower()
        if not q or q in searchable_content:
            matched_models.append(m)
    
    if platform_type == "ollama":
        options = [(f"{m['name']} ({m['size']}) [{m['category']}]", m['id']) for m in matched_models]
        options.append(("➕ Benutzerdefiniertes Modell (Tag eingeben)", "CUSTOM_OLLAMA"))
    else:
        options = [(f"{m['name']} [{m['category']}] ({m['popularity']})", m['id']) for m in matched_models]
        options.append(("➕ Benutzerdefiniertes Hugging Face Modell (Repo eingeben)", "CUSTOM_HF"))
    
    if matched_models or q == "":
        dropdown_model.options = options
        dropdown_model.disabled = False
        btn_start_download.disabled = False
        if options:
            dropdown_model.value = options[0][1]
            if platform_type == "huggingface" and options[0][1] != "CUSTOM_HF":
                update_hf_file_dropdown(options[0][1])
            matched_item = next((m for m in CURRENT_LOADED_MODELS if m['id'] == options[0][1]), None)
            html_spec_card.value = render_spec_card(matched_item)
    else:
        dropdown_model.options = [(f"Keine Treffer für '{query}'", "")]
        dropdown_model.disabled = True
        btn_start_download.disabled = True
        html_spec_card.value = "<div style='padding: 10px; color: #DC2626; font-size: 13px;'>Keine passenden Modelle gefunden. Bitte Suchbegriff anpassen.</div>"

def on_search_query_changed(change):
    filter_model_options(change.new if change.new else "")

text_model_search.observe(on_search_query_changed, names='value')

def on_platform_confirm_clicked(b):
    global CURRENT_LOADED_MODELS
    platform_type = dropdown_platform.value
    label_status.value = f"Lade Live-Modelle für '{platform_type}'..."
    btn_confirm_platform.disabled = True
    dropdown_model.disabled = True
    text_model_search.value = "" # Suchfeld zurücksetzen
    
    try:
        if platform_type == "ollama":
            CURRENT_LOADED_MODELS = ollama_backend.fetch_models(limit=80)
            dropdown_gguf_file.layout.display = "none"
            text_custom_hf_repo.layout.display = "none"
            text_custom_hf_file.layout.display = "none"
        else:
            CURRENT_LOADED_MODELS = hf_backend.fetch_models(limit=50)
            dropdown_gguf_file.layout.display = "block"
        
        filter_model_options("")
        label_status.value = f"{len(CURRENT_LOADED_MODELS)} Live-Modelle für '{platform_type.upper()}' bereit."
    except Exception as err:
        label_status.value = f"Fehler beim Laden: {err}"
    finally:
        btn_confirm_platform.disabled = False

btn_confirm_platform.on_click(on_platform_confirm_clicked)

def update_hf_file_dropdown(repo_id: str):
    files = hf_backend.fetch_gguf_files(repo_id)
    if files:
        dropdown_gguf_file.options = [(f, f) for f in files]
        dropdown_gguf_file.disabled = False
    else:
        dropdown_gguf_file.options = [("Keine .gguf-Dateien gefunden", "")]
        dropdown_gguf_file.disabled = True

def on_model_selection_changed(change):
    if not change.new:
        return
    selected_id = change.new
    
    if selected_id == "CUSTOM_OLLAMA":
        text_custom_tag.layout.display = "block"
        text_custom_hf_repo.layout.display = "none"
        text_custom_hf_file.layout.display = "none"
        dropdown_gguf_file.layout.display = "none"
    elif selected_id == "CUSTOM_HF":
        text_custom_tag.layout.display = "none"
        text_custom_hf_repo.layout.display = "block"
        text_custom_hf_file.layout.display = "block"
        dropdown_gguf_file.layout.display = "none"
    else:
        text_custom_tag.layout.display = "none"
        text_custom_hf_repo.layout.display = "none"
        text_custom_hf_file.layout.display = "none"
        
        if dropdown_platform.value == "huggingface":
            dropdown_gguf_file.layout.display = "block"
            update_hf_file_dropdown(selected_id)
        else:
            dropdown_gguf_file.layout.display = "none"
            
        matched = next((m for m in CURRENT_LOADED_MODELS if m['id'] == selected_id), None)
        if matched:
            html_spec_card.value = render_spec_card(matched)

dropdown_model.observe(on_model_selection_changed, names='value')

# ==============================================================================
# DOWNLOAD EXECUTION
# ==============================================================================
def run_download_process():
    platform_choice = dropdown_platform.value
    selected_id = dropdown_model.value
    
    # UI sperren
    btn_start_download.disabled = True
    btn_confirm_platform.disabled = True
    dropdown_platform.disabled = True
    text_model_search.disabled = True
    dropdown_model.disabled = True
    dropdown_gguf_file.disabled = True
    progress_bar.layout.display = "block"
    progress_bar.value = 0
    label_status.value = "Starte Download..."
    
    with output_log:
        clear_output()
        print("=" * 60)
        print(f"--- START DOWNLOAD & INTEGRATION ({platform_choice.upper()}) ---")
        print(f"Zeit: {time.strftime('%Y-%m-%d %H:%M:%S')}")
        print("=" * 60)
    
    try:
        if platform_choice == "ollama":
            target_tag = text_custom_tag.value.strip() if selected_id == "CUSTOM_OLLAMA" else selected_id
            if not target_tag:
                raise ValueError("Bitte einen gültigen Ollama-Modelltag angeben!")
            
            with output_log:
                print(f"[Ollama Pull] Lade '{target_tag}'...")
            
            current_digest = None
            def ollama_callback(progress):
                nonlocal current_digest
                status = progress.get('status', '')
                completed = progress.get('completed', 0)
                total = progress.get('total', 0)
                digest = progress.get('digest', '')
                
                if digest != current_digest and digest:
                    with output_log:
                        print(f"[Layer {digest[:12]}] {status}")
                    current_digest = digest
                
                if total and completed and total > 0:
                    pct = min(int((completed / total) * 100), 100)
                    progress_bar.value = pct
                    label_status.value = f"[Pull] {status} ({completed // (1024**2)} / {total // (1024**2)} MB)"
                else:
                    label_status.value = f"[Pull] {status}"
            
            ollama_backend.pull_model(target_tag, progress_callback=ollama_callback)
            registered_name = target_tag
            allocated_size = 4.0
            
        else: # Hugging Face
            if selected_id == "CUSTOM_HF":
                repo_id = text_custom_hf_repo.value.strip()
                filename = text_custom_hf_file.value.strip()
            else:
                repo_id = selected_id
                filename = dropdown_gguf_file.value
            
            if not repo_id or not filename:
                raise ValueError("Bitte Repo-ID und GGUF-Dateinamen angeben!")
            
            target_tag = filename.lower().replace(".gguf", "").replace("_", "-").replace(".", "-")
            
            def hf_progress(msg):
                label_status.value = msg
                with output_log:
                    print(f"[HF Hub] {msg}")
            
            def hf_build(st):
                label_status.value = f"[Ollama Build] {st}"
                with output_log:
                    print(f"[Ollama Build] {st}")
            
            progress_bar.value = 30
            local_path, allocated_size = hf_backend.download_and_create_model(
                repo_id=repo_id,
                filename=filename,
                target_model_tag=target_tag,
                dest_dir=HF_MODELS_DIR,
                progress_callback=hf_progress,
                build_callback=hf_build
            )
            registered_name = target_tag
        
        # Konfiguration sichern
        ModelConfigManager.save_active_model(
            model_name=registered_name,
            size_gb=allocated_size,
            platform_name=platform_choice,
            config_dir=CONFIG_DIR
        )
        
        progress_bar.value = 100
        label_status.value = f"[✓] Erfolgreich! '{registered_name}' ist offline einsatzbereit."
        with output_log:
            print("=" * 60)
            print(f"[ERFOLG] Modell '{registered_name}' erfolgreich in Ollama registriert.")
            print(f"Gespeichert in active_model_config.json")
            print("=" * 60)
            
    except Exception as err:
        progress_bar.bar_style = "danger"
        label_status.value = f"[FEHLER] {err}"
        with output_log:
            print(f"\n[FEHLER] {err}")
    finally:
        btn_start_download.disabled = False
        btn_confirm_platform.disabled = False
        dropdown_platform.disabled = False
        text_model_search.disabled = False
        dropdown_model.disabled = False
        dropdown_gguf_file.disabled = False

def on_start_download_clicked(b):
    t = threading.Thread(target=run_download_process)
    t.daemon = True
    t.start()

btn_start_download.on_click(on_start_download_clicked)

# ==============================================================================
# DASHBOARD CONTAINER
# ==============================================================================
ui_container = widgets.VBox([
    widgets.HTML("<h4>🎛️ Dynamische Modell-Verwaltung</h4>"),
    widgets.HBox([dropdown_platform, btn_confirm_platform]),
    widgets.HTML("<hr style='margin: 6px 0; border: 0; border-top: 1px solid #E5E7EB;'>"),
    text_model_search,
    dropdown_model,
    dropdown_gguf_file,
    text_custom_tag,
    widgets.HBox([text_custom_hf_repo, text_custom_hf_file]),
    html_spec_card,
    btn_start_download,
    progress_bar,
    label_status,
    widgets.HTML("<div style='margin-top: 6px; font-weight: bold; font-size: 11px; color: #6B7280;'>Prozessprotokoll:</div>"),
    output_log
], layout=widgets.Layout(padding="14px", border="1px solid #D1D5DB", border_radius="6px", background_color="#FFFFFF"))

# Erste Initialisierung
on_platform_confirm_clicked(None)
display(ui_container)

### 🧪 Schritt 4: Modell-Inspektion & Offline-Test
Auflistung der lokal registrierten Ollama-Modelle und Ausführung eines Test-Prompts zur Verifikation.

In [ ]:
# Lokal registrierte Modelle abrufen
local_models = service_manager.client.list()

# Aktive Konfiguration ermitteln
active_cfg_file = os.path.join(CONFIG_DIR, "active_model_config.json")
active_model_tag = "llama3.1:latest"
if os.path.exists(active_cfg_file):
    with open(active_cfg_file, "r", encoding="utf-8") as f:
        cfg = json.load(f)
        active_model_tag = cfg.get("model_name", active_model_tag)

print("=" * 60)
print("=== LOKALE OLLAMA-MODELLE ===")
for i, item in enumerate(local_models.models, 1):
    size_mb = item.size // (1024**2) if item.size else 0
    active_flag = " (AKTIV)" if item.model.startswith(active_model_tag.split(":")[0]) else ""
    print(f" {i}. {item.model:<35} | {size_mb:>6} MB{active_flag}")
print("=" * 60)

# Test-Inferenz
prompt = "Erkläre kurz in einem Satz den Vorteil lokaler KI-Modelle."
print(f"\n[TEST-INFERENZ] Modell: '{active_model_tag}'")
print(f"Prompt: '{prompt}'\n")

try:
    res = service_manager.client.generate(model=active_model_tag, prompt=prompt)
    print("--- ANTWORT ---")
    print(res.get('response', '').strip())
    print("----------------")
    print("[✓] Offline-Inferenz erfolgreich!")
except Exception as err:
    print(f"[Hinweis] Test nicht ausgeführt: {err}")

---
### 🔄 Nächste Schritte
Das Modell ist nun offline in Ollama bereitgestellt und in `config/active_model_config.json` hinterlegt.

* 👉 **[02_dockereinstellung.ipynb](file:notebooks/02_dockereinstellung.ipynb)** (Traefik-Gateway & Multi-User Isolation)
* 👉 **[03_html_embed.ipynb](file:notebooks/03_html_embed.ipynb)** (Streamlit Web-UI & HTML Embedding)
* 👉 **[Srart_mai_ai.ipynb](file:notebooks/Srart_mai_ai.ipynb)** (Hauptsystem-Start)